# SYNTHETIC TECHNICAL VALIDATION — NOT BAF RESULTS
960 generated rows; reduced budgets; two CPU seeds. This checks execution and evidence, not research performance.

# 15 — Imbalance-XGBoost

Import trực tiếp gradient/Hessian weighted và focal từ commit tác giả, chạy qua API xgb.train hiện tại. So với XGB không weight cùng preprocessing/budget. Early stopping theo unweighted validation AP.

Không sửa cấu hình ở notebook này. Muốn đổi cấu hình hãy tạo campaign mới ở11.

In [1]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src" / "baf.py").is_file():
    raise RuntimeError("Mở notebook từ thư mục BAF_project hoặc BAF_project/notebooks.")

CACHE_ROOT = ROOT.parent.parent / ".cache" / "baf-research"
if os.name == "nt":
    CACHE_ROOT = Path(r"D:\UIT\.cache\baf-research")
for name, subdirectory in {
    "PIP_CACHE_DIR": "pip",
    "TORCH_HOME": "torch",
    "XDG_CACHE_HOME": "misc",
    "TEMP": "temp",
    "TMP": "temp",
}.items():
    folder = CACHE_ROOT / subdirectory
    folder.mkdir(parents=True, exist_ok=True)
    os.environ[name] = str(folder)

sys.path.insert(0, str(ROOT / "src"))
import gru

print("Python:", sys.executable)
print("Project:", ROOT)
print("Module:", gru.RESEARCH_VERSION)


Python: /opt/codex/runtimes/codex-primary-runtime/dependencies/python/bin/python3
Project: /workspace/scratch/a3538503433e/work/research11/synthetic_validation
Module: baf-research-20261028-v1


In [2]:
# Điền đường dẫn in ở notebook 11 nếu đã có nhiều campaign.
# Ví dụ: Path(r"D:\UIT\DS317\BAF_project\results\research_campaigns\weighting_research_...")
CAMPAIGN_DIR = None
CAMPAIGN = gru.research_resolve_campaign(ROOT, CAMPAIGN_DIR)
PLAN = gru.research_read(CAMPAIGN / "campaign.json")

print("Campaign:", CAMPAIGN)
print("Mode:", PLAN["mode"])
print("Suites:", PLAN["suites"])
print("Seeds:", PLAN["config"]["seeds"])


Campaign: /workspace/scratch/a3538503433e/work/research11/synthetic_validation/results/research_campaigns/weighting_research_20261028_142640_1f37cc05
Mode: synthetic
Suites: ['losses', 'meta', 'lsr', 'xgboost']
Seeds: [42, 317]


## 1. Chạy training có notebook output thật

        Cell khởi chạy một Python process mới bằng đúng interpreter của kernel. Process thực thi từng cell trong notebook con, lưu stdout/display/error sau từng cell và in log trực tiếp bên dưới. Fit hoàn tất được kiểm hash rồi reuse khi resume. Nếu bị ngắt, không xóa campaign. Cell này có thể chạy lâu.

In [3]:
SUITE = 'xgboost'
RUN_DIR = None
if SUITE in PLAN["suites"]:
    RUN_DIR = gru.research_launch_suite(CAMPAIGN, SUITE)
else:
    print("SKIPPED: suite không thuộc phạm vi đã chọn ở11:", SUITE)


BEGIN_STAGE 1
END_STAGE 1
BEGIN_STAGE 2
RESEARCH_STUDY xgboost mode= synthetic device= cpu
END_STAGE 2
BEGIN_STAGE 3
FIT_BEGIN xgb_none_c00_s42 {'method': 'xgb_none', 'family': 'xgboost', 'candidate': 0, 'depth': 3, 'gamma': 2.0, 'alpha': 1.5549540505455315, 'feature': 'dense'}
[0]	validation-unweighted_AP:0.21128
[9]	validation-unweighted_AP:0.16791
FIT_COMPLETE xgb_none_c00_s42 validation_AP= 0.29879426129426134
FIT_BEGIN xgb_weighted_c00_s42 {'method': 'xgb_weighted', 'family': 'xgboost', 'candidate': 0, 'depth': 3, 'gamma': 2.0, 'alpha': 1.5549540505455315, 'feature': 'dense'}
[0]	validation-unweighted_AP:0.14566
[9]	validation-unweighted_AP:0.17018
FIT_COMPLETE xgb_weighted_c00_s42 validation_AP= 0.19590461966840952
FIT_BEGIN xgb_focal_c00_s42 {'method': 'xgb_focal', 'family': 'xgboost', 'candidate': 0, 'depth': 3, 'gamma': 2.0, 'alpha': 1.5549540505455315, 'feature': 'dense'}
[0]	validation-unweighted_AP:0.18882
[9]	validation-unweighted_AP:0.18828
FIT_COMPLETE xgb_focal_c00_s42 

## 2. Xem kết quả validation đã khóa

        Chưa có test metric ở notebook này. Model, configuration và threshold được lưu trước khi mở test ở16.

In [4]:
if RUN_DIR is not None:
    display(gru.research_suite_overview(RUN_DIR))
    print("Notebook training thật:")
    for executed in sorted((RUN_DIR / "attempts").glob("*/training_executed.ipynb")):
        print(executed)


Notebook training thật:
/workspace/scratch/a3538503433e/work/research11/synthetic_validation/results/research_campaigns/weighting_research_20261028_142640_1f37cc05/suites/xgboost/attempts/20261028_142722_87d1a02f/training_executed.ipynb


,method,seed,validation_ap,threshold,fit_seconds,family,feature
0,xgb_focal,42,0.231526,0.489160,0.041292,xgboost,dense
1,xgb_focal,317,0.233597,0.466232,0.036612,xgboost,dense
2,xgb_none,42,0.298794,0.484321,0.081422,xgboost,dense
3,xgb_none,317,0.233728,0.480185,0.027453,xgboost,dense
4,xgb_weighted,42,0.195905,0.475646,0.038457,xgboost,dense
5,xgb_weighted,317,0.210066,0.473008,0.033525,xgboost,dense


## 3. Xuất proof trong một lượt

        Không cần Ctrl+S hoặc chạy cell cuối hai lần. Full proof giữ trên máy; review nhỏ hơn để gửi, ghi rõ phần đã bỏ. Notebook `verify.ipynb` được execute thật, kiểm hash/score và không fit lại.

In [5]:
if RUN_DIR is not None:
    PROOF = gru.research_export_proof(RUN_DIR, ROOT)
    for kind, path in PROOF.items():
        print(kind, "→", path)


RESEARCH_FROZEN_SUITE_VERIFIED 6
FULL_PROOF /workspace/scratch/a3538503433e/work/research11/synthetic_validation/results/proofs/xgboost_8a707624_full_proof.zip
PARTIAL_REVIEW /workspace/scratch/a3538503433e/work/research11/synthetic_validation/results/proofs/xgboost_8a707624_review.zip
full_proof → /workspace/scratch/a3538503433e/work/research11/synthetic_validation/results/proofs/xgboost_8a707624_full_proof.zip
review → /workspace/scratch/a3538503433e/work/research11/synthetic_validation/results/proofs/xgboost_8a707624_review.zip
